# Семинар 4. Анализ тональности текстов (NLP)

**Цель семинара:** Научиться извлекать количественные аналитические метрики из неструктурированных текстовых данных (отзывов, жалоб, тикетов поддержки). Мы освоим базовую очистку текстов, подключим мощную предобученную ИИ-модель через библиотеку `transformers`, преобразуем текстовые эмоции в строгий математический скор и сагрегируем его до уровня клиента.

### 🔧 Настройка окружения и импорт библиотек

Для работы с глубоким обучением нам понадобится экосистема Hugging Face. Если библиотека `transformers` не установлена, установите её через ваш менеджер пакетов (`uv add transformers`).


In [ ]:
import os
import re
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

# Подавляем предупреждения от Hugging Face
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 
warnings.filterwarnings('ignore')

try:
    from transformers import pipeline
except ImportError:
    print("⚠️ Библиотека transformers не найдена. Установите её: uv add transformers torch")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


---

## 📥 Шаг 1. Инициализация локального контура (Трек В: Отзывы)

Загружаем третий источник данных — выгрузку неструктурированных текстовых отзывов `reviews.csv`.


In [ ]:
INPUT_DIR = os.path.abspath(os.path.join(".", "data", "input"))
OUTPUT_DIR = os.path.abspath(os.path.join(".", "data", "seminar_4_nlp_sentiment"))
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

reviews_csv_path = os.path.abspath(os.path.join(INPUT_DIR, "reviews.csv"))
nlp_csv_path = os.path.abspath(os.path.join(OUTPUT_DIR, "nlp_features.csv"))

if not os.path.exists(reviews_csv_path):
    print(f"⚠️ Файл отзывов не найден: {reviews_csv_path}. Генерируется mock-датасет.")
    
    # Генерация синтетических отзывов
    mock_reviews = pd.DataFrame({
        'Review_ID': ['R001', 'R002', 'R003', 'R004', 'R005'],
        'Target_ID': ['T01', 'T01', 'T02', 'T03', 'T04'],
        'Review_Date': ['2023-10-01', '2023-10-15', '2023-11-02', '2023-11-20', '2023-12-05'],
        'Review_Text': [
            'Ужасный сервис, постоянные обрывы сети! Я очень зол!!! 😡',
            'Верните деньги за подписку, ни за что с вами больше не свяжусь.',
            'В целом нормально, но поддержка отвечает долго.',
            'Отличная скорость, очень вежливые операторы. Спасибо!',
            'Neutral comment about the product features.'
        ]
    })
    mock_reviews.to_csv(reviews_csv_path, index=False)

print(f"Путь к логу отзывов инициализирован: {reviews_csv_path}")
print(f"Путь к итоговым NLP фичам: {nlp_csv_path}")


---

## 🛠 ЗАДАНИЕ 1: Базовая очистка неструктурированного текста
**Бизнес-контекст:** Текст из интернета полон мусора: знаки препинания, эмодзи, лишние пробелы, разный регистр. Прежде чем отдавать текст модели, его нужно привести к единому стандарту, чтобы снизить вычислительную нагрузку и вероятность ошибок.

**Инструкция (TODO):**
1. Считайте файл в датафрейм `df_reviews`.
2. Создайте новую колонку `Clean_Text`.
3. Приведите исходный текст к нижнему регистру.
4. Удалите все небуквенные и нецифровые символы (пунктуацию, эмодзи и спецсимволы), заменив их на пробелы.
5. Схлопните множественные пробелы в один и обрежьте лишние пробелы по краям строки.

*🤖 Теги для AI-ментора: `#SEM4_TASK1_START`, `#SEM4_TASK1_BUG`*


In [ ]:
# [MASTER SOLUTION]
df_reviews = pd.read_csv(reviews_csv_path)

# Базовая очистка текста
df_reviews['Clean_Text'] = df_reviews['Review_Text'].astype(str).str.lower()
# Оставляем только буквы, цифры и пробелы
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(r'[^\w\s]', ' ', regex=True)
# Убираем двойные/тройные пробелы
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(r'\s+', ' ', regex=True).str.strip()

display(df_reviews[['Review_Text', 'Clean_Text']].head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Загрузите датасет
# TODO: 1.2. Создайте колонку Clean_Text, переведите в нижний регистр
# TODO: 1.3. Примените регулярные выражения для удаления пунктуации и лишних пробелов
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_reviews = pd.read_csv(...)

df_reviews['Clean_Text'] = df_reviews['Review_Text'].astype(str)....
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(..., ' ', regex=True)
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(..., ' ', regex=True).str.strip()


---

## 🛠 ЗАДАНИЕ 2 и 3: Инференс Трансформера и Математический Маппинг
**Бизнес-контекст:** Мы используем предобученную нейросеть `tabularisai/multilingual-sentiment-analysis`. Она читает текст и возвращает категорию (например, "1 star", "5 stars" или "Positive", "Negative"). Модель машинного обучения для предсказания оттока не поймет эти слова. Нам нужно смаппировать их в строгий математический интервал: от `-1.0` (глубокий негатив) до `+1.0` (позитив). Нейтральные отзывы получат `0.0`.

**Инструкция (TODO):**
1. Инициализируйте пайплайн `classifier = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", model_kwargs={"cache_dir": MODELS_DIR})`.
2. Напишите функцию `calculate_score(text)`, которая прогоняет текст через модель и возвращает число от -1.0 до 1.0 на основе метки. *(Подсказка: модель обычно возвращает словарь `[{'label': '5 stars', 'score': 0.9}]`. Маппинг: 1/2 stars -> -1.0, 3 stars -> 0.0, 4/5 stars -> 1.0).*
3. Примените функцию к колонке `Clean_Text` и создайте новую фичу `Sentiment_Score`.

*🤖 Теги для AI-ментора: `#SEM4_TASK2_START`, `#SEM4_TASK2_WHY`*


In [ ]:
# [MASTER SOLUTION]
try:
    print("⏳ Инициализация ИИ-модели (это может занять время)...")
    sentiment_pipeline = pipeline(
        "text-classification",
        model="tabularisai/multilingual-sentiment-analysis",
        model_kwargs={"cache_dir": MODELS_DIR}
    )
    
    def calculate_score(text):
        if not text or pd.isna(text) or str(text).strip() == '':
            return 0.0
            
        result = sentiment_pipeline(str(text)[:512])[0] # Ограничение трансформеров на 512 токенов
        label = str(result['label']).lower()
        
        # Маппинг меток в интервал [-1.0, 1.0]
        if any(k in label for k in ['1', '2', 'neg']):
            return -1.0
        elif any(k in label for k in ['4', '5', 'pos']):
            return 1.0
        else:
            return 0.0 # Neutral or 3 stars
            
except Exception as e:
    print(f"⚠️ Не удалось загрузить трансформер: {e}. Используем эвристическую фоллбэк-функцию.")
    def calculate_score(text):
        if not text or pd.isna(text) or str(text).strip() == '':
            return 0.0
        t = str(text).lower()
        if any(w in t for w in ['ужас', 'зол', 'верните', 'плох', 'bad', 'poor']): return -1.0
        if any(w in t for w in ['отличн', 'спасибо', 'хорош', 'good', 'great']): return 1.0
        return 0.0

# Применение к датафрейму
print("⏳ Анализ тональности отзывов...")
df_reviews['Sentiment_Score'] = df_reviews['Clean_Text'].apply(calculate_score)

display(df_reviews[['Clean_Text', 'Sentiment_Score']].head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Инициализируйте пайплайн Hugging Face
# TODO: 2.2. Напишите логику функции calculate_score для конвертации текстовой метки в float
# TODO: 2.3. Примените функцию к столбцу Clean_Text с помощью .apply()
raise NotImplementedError("Задания 2 и 3 не выполнены! Удалите эту строку и напишите свой код.")

sentiment_pipeline = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", model_kwargs={"cache_dir": MODELS_DIR})

def calculate_score(text):
    if not text or str(text).strip() == '':
        return 0.0
    
    result = sentiment_pipeline(str(text)[:512])[0]
    label = str(result['label']).lower()
    
    # TODO: Реализуйте if/else маппинг. 
    # Верните -1.0 для негатива, 1.0 для позитива, 0.0 для нейтрального.
    ...

df_reviews['Sentiment_Score'] = df_reviews['Clean_Text'].apply(...)


---

## 🛠 ЗАДАНИЕ 4: Агрегация текстовых метрик до уровня клиента
**Бизнес-контекст:** У одного клиента (`Target_ID`) может быть 5 разных отзывов. Нам нужно свернуть их в единую характеристику лояльности этого субъекта, чтобы позже прикрепить её к мастер-профилю. Мы усредним оценки.

**Инструкция (TODO):**
1. Сгруппируйте `df_reviews` по ключу `Target_ID`.
2. Рассчитайте среднее (`mean`) для колонки `Sentiment_Score`.
3. Сбросьте индекс и переименуйте колонку в `Mean_Sentiment`.


In [ ]:
# [MASTER SOLUTION]
df_agg_sentiment = df_reviews.groupby('Target_ID')['Sentiment_Score'].mean().reset_index()
df_agg_sentiment.rename(columns={'Sentiment_Score': 'Mean_Sentiment'}, inplace=True)

print("Агрегированные оценки тональности по клиентам:")
display(df_agg_sentiment.head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 4.1. Используйте groupby('Target_ID') для колонки 'Sentiment_Score' и вычислите mean()
# TODO: 4.2. Сделайте reset_index() и переименуйте результат
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

df_agg_sentiment = df_reviews.groupby(...)[...].mean().reset_index()
df_agg_sentiment.rename(columns={...: 'Mean_Sentiment'}, inplace=True)

display(df_agg_sentiment.head())


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция extract_sentiment_features

Упакуем наши наработки в production-функцию `extract_sentiment_features`. Она принимает датафрейм, названия колонок и имя модели, динамически настраивает локальное кэширование весов и рассчитывает чистый агрегированный сентимент.


In [ ]:
# [MASTER SOLUTION]
def extract_sentiment_features(
    df: pd.DataFrame, 
    text_col: str, 
    id_col: str, 
    model_name: str = "tabularisai/multilingual-sentiment-analysis"
) -> pd.DataFrame:
    """
    Конвейер извлечения NLP-метрик тональности. Очищает текст, запускает 
    предобученный трансформер с локальным кэшированием весов, маппит метки и агрегирует результат.
    """
    df_pipe = df.copy()
    
    # 1. Очистка текста
    clean_col = 'Clean_Text_Tmp'
    df_pipe[clean_col] = df_pipe[text_col].astype(str).str.lower()
    df_pipe[clean_col] = df_pipe[clean_col].str.replace(r'[^\w\s]', ' ', regex=True)
    df_pipe[clean_col] = df_pipe[clean_col].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # 2. Инициализация и Инференс Transformer
    try:
        cache_dir = os.path.join(OUTPUT_DIR, "models")
        os.makedirs(cache_dir, exist_ok=True)
        nlp_pipeline = pipeline(
            "text-classification",
            model=model_name,
            model_kwargs={"cache_dir": cache_dir}
        )
        
        def _score(text):
            if not text or pd.isna(text) or str(text).strip() == '':
                return 0.0
            res = nlp_pipeline(str(text)[:512])[0]
            lbl = str(res['label']).lower()
            if any(k in lbl for k in ['1', '2', 'neg']):
                return -1.0
            elif any(k in lbl for k in ['4', '5', 'pos']):
                return 1.0
            return 0.0
    except Exception as e:
        print(f"⚠️ Не удалось запустить трансформер ({e}). Применяем эвристический фоллбэк.")
        def _score(text):
            if not text or pd.isna(text) or str(text).strip() == '':
                return 0.0
            t = str(text).lower()
            if any(w in t for w in ['ужас', 'зол', 'верните', 'плох', 'bad', 'poor']):
                return -1.0
            if any(w in t for w in ['отличн', 'спасибо', 'хорош', 'good', 'great']):
                return 1.0
            return 0.0
        
    df_pipe['Sentiment_Score'] = df_pipe[clean_col].apply(_score)
    
    # 3. Агрегация
    df_result = df_pipe.groupby(id_col)['Sentiment_Score'].mean().reset_index()
    df_result.rename(columns={'Sentiment_Score': 'Mean_Sentiment'}, inplace=True)
    
    return df_result

# Тестируем собранный конвейер
df_nlp_final = extract_sentiment_features(
    df=pd.read_csv(reviews_csv_path),
    text_col='Review_Text',
    id_col='Target_ID'
)
print("Пайплайн NLP успешно завершен!")
df_nlp_final.to_csv(nlp_csv_path, index=False, encoding='utf-8')
print(f"Результат успешно выгружен и сохранен в: {nlp_csv_path}")
display(df_nlp_final.head())


In [ ]:
# [STUDENT TEMPLATE]
def extract_sentiment_features(
    df: pd.DataFrame, 
    text_col: str, 
    id_col: str, 
    model_name: str = "tabularisai/multilingual-sentiment-analysis"
) -> pd.DataFrame:
    """
    Промышленная функция для извлечения метрик тональности.
    """
    # TODO: Соберите архитектуру функции, опираясь на параметры text_col и id_col
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    df_pipe = df.copy()
    
    # 1. Очистка текста
    # ...
    
    # 2. Модель и Маппинг
    # ...
    
    # 3. Агрегация
    # ...
    
    return pd.DataFrame()

df_nlp_final = extract_sentiment_features(pd.read_csv(reviews_csv_path), 'Review_Text', 'Target_ID')
df_nlp_final.to_csv(nlp_csv_path, index=False, encoding='utf-8')
print(f"Результат успешно выгружен и сохранен в: {nlp_csv_path}")


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Запустите скрипт проверки, чтобы убедиться, что ваша NLP-функция возвращает валидный датафрейм и успешно сохраняет файл на диск.


In [ ]:
def run_autocheck(file_path: str = nlp_csv_path):
    print(f"🚀 Загрузка и тестирование итогового файла: {file_path}\n" + "-"*45)
    validation_status = True
    
    if not os.path.exists(file_path):
        print(f"❌ Ошибка: Файл по пути '{file_path}' не найден!")
        print("   Убедитесь, что вы применили метод .to_csv() с правильным указанием папки.")
        return False
        
    try:
        df_to_check = pd.read_csv(file_path, encoding='utf-8')
        print(f"📊 Файл успешно считан. Размерность таблицы: {df_to_check.shape}")
    except Exception as e:
        print(f"❌ Ошибка при чтении CSV-файла: {e}")
        return False
        
    expected_cols = ['Target_ID', 'Mean_Sentiment']
    missing_cols = [c for c in expected_cols if c not in df_to_check.columns]
    
    # Проверка структуры столбцов
    if missing_cols:
        print(f"❌ Ошибка: В таблице отсутствуют обязательные столбцы: {missing_cols}")
        validation_status = False
    else:
        print("✅ Колонки Target_ID и Mean_Sentiment присутствуют.")
        
    # Проверка типов и границ сентимента
    if 'Mean_Sentiment' in df_to_check.columns:
        if not pd.api.types.is_numeric_dtype(df_to_check['Mean_Sentiment']):
            print("❌ Ошибка: Колонка Mean_Sentiment имеет нечисловой тип.")
            validation_status = False
        else:
            min_val = df_to_check['Mean_Sentiment'].min()
            max_val = df_to_check['Mean_Sentiment'].max()
            if min_val < -1.01 or max_val > 1.01:
                print(f"❌ Ошибка: Нарушены границы сентимента [-1, 1]. Найдены значения: {min_val} до {max_val}")
                validation_status = False
            else:
                print("✅ Математический маппинг выполнен в корректных границах [-1.0, 1.0].")
                
    # Проверка гранулярности
    if 'Target_ID' in df_to_check.columns:
        if df_to_check['Target_ID'].duplicated().sum() > 0:
            print("❌ Ошибка: Нарушена агрегация. У одного клиента найдено несколько строк.")
            validation_status = False
        else:
            print("✅ Группировка проведена верно, строго одна оценка тональности на клиента.")

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! NLP-пайплайн полностью готов.")
        print("Добавьте эту функцию в файл course_project/src/data_pipeline.py!")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию вашей функции.")

run_autocheck()